# RSI Backtest Report

**Scope:** 21 FX pairs · RSI offset 17 · costs excluded.

Results use closed trades exported by TradingView. Open positions are retained in the merged CSV and excluded from realized performance. This report assesses the supplied rules; it does not independently rerun the strategy on candle data.

## Data preparation

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import hashlib
import itertools
project_folder = Path.cwd()
if project_folder.name == 'final':
    project_folder = project_folder.parent
input_folder = project_folder / 'xlsxs'
output_folder = project_folder / 'final'
output_folder.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 220)
files = sorted(input_folder.glob('*.xlsx'))
assert len(files) == 21, 'تعداد فایل\u200cها باید ۲۱ باشه.'
all_trades = []
file_checks = []
settings = []
trade_hashes = set()
for file in files:
    sheets = pd.read_excel(file, sheet_name=None)
    required = {'Properties', 'Trades', 'Performance', 'Trades analysis', 'Risk-adjusted performance'}
    assert required.issubset(sheets), f'شیت ناقص: {file.name}'
    properties_sheet = pd.read_excel(file, sheet_name='Properties', keep_default_na=False)
    props = dict(properties_sheet.iloc[:, :2].itertuples(index=False, name=None))
    symbol = props['Symbol']
    raw = sheets['Trades']
    digest = hashlib.sha256(raw.iloc[:, :].drop(columns=[raw.columns[4]]).to_csv(index=False).encode()).hexdigest()
    assert digest not in trade_hashes, f'محتوای تکراری: {file.name}'
    trade_hashes.add(digest)
    assert raw['Trade number'].notna().all()
    entries = raw[raw['Type'].str.startswith('Entry', na=False)].copy()
    exits = raw[raw['Type'].str.startswith('Exit', na=False)].copy()
    assert len(entries) + len(exits) == len(raw), f'نوع ردیف ناشناخته: {symbol}'
    exits = exits[exits['Date and time'].astype(str).ne('Open')].copy()
    assert not entries['Trade number'].duplicated().any()
    assert not exits['Trade number'].duplicated().any()
    assert set(exits['Trade number']).issubset(set(entries['Trade number']))
    price_column = raw.columns[4]
    joined = entries.merge(exits, on='Trade number', how='left', suffixes=('_entry', '_exit'), validate='one_to_one')
    table = pd.DataFrame({'symbol': symbol, 'trade_number': joined['Trade number'].astype(int), 'direction': joined['Type_entry'].str.replace('Entry ', '', regex=False), 'entry_time': pd.to_datetime(joined['Date and time_entry'], errors='raise'), 'exit_time': pd.to_datetime(joined['Date and time_exit'], errors='raise'), 'entry_price': joined[price_column + '_entry'], 'exit_price': joined[price_column + '_exit'], 'quantity': joined['Size (qty)_entry'], 'entry_value_quote': joined['Size (value)_entry'], 'pnl_usd': joined['Net PnL USD_exit'], 'trade_return_pct': joined['Return %_exit'], 'commission_usd': joined['Commission USD_exit'], 'mfe_usd': joined['Favorable excursion USD_exit'], 'mae_usd': joined['Adverse excursion USD_exit'], 'duration_bars': joined['Duration (bars)_exit'], 'tv_cumulative_pnl_usd': joined['Cumulative PnL USD_exit'], 'exit_signal': joined['Signal_exit'], 'quote_currency': props['Currency'], 'source_file': file.name})
    table['status'] = np.where(table['exit_time'].notna(), 'closed', 'open')
    table['trade_id'] = table['symbol'] + ':' + table['trade_number'].astype(str)
    table = table.sort_values('entry_time').reset_index(drop=True)
    closed = table[table['status'] == 'closed']
    assert table[['entry_time', 'entry_price', 'quantity', 'direction']].notna().all().all()
    assert closed[['exit_price', 'pnl_usd', 'duration_bars']].notna().all().all()
    assert (closed['exit_time'] >= closed['entry_time']).all()
    assert (table['entry_time'].iloc[1:].reset_index(drop=True) >= table['exit_time'].iloc[:-1].reset_index(drop=True)).all(), f'هم\u200cپوشانی یا معاملهٔ باز میان تاریخچه: {symbol}'
    direction_sign = np.where(closed['direction'] == 'long', 1, -1)
    price_pnl = (closed['exit_price'] - closed['entry_price']) * direction_sign
    assert (np.sign(price_pnl) == np.sign(closed['pnl_usd'])).all()
    analysis = sheets['Trades analysis'].set_index(sheets['Trades analysis'].columns[0])
    performance = sheets['Performance'].set_index(sheets['Performance'].columns[0])
    reported_count = int(analysis.loc['Total trades'].iloc[0])
    reported_open = int(analysis.loc['Total open trades'].iloc[0])
    reported_pnl = float(performance.loc['Net profit'].iloc[0])
    assert len(closed) == reported_count
    assert (table['status'] == 'open').sum() == reported_open
    assert (closed['pnl_usd'] > 0).sum() == int(analysis.loc['Total winners'].iloc[0])
    assert (closed['pnl_usd'] < 0).sum() == int(analysis.loc['Total losers'].iloc[0])
    pnl_difference = closed['pnl_usd'].sum() - reported_pnl
    assert abs(pnl_difference) <= 0.005 * len(closed) + 0.011, f'اختلاف فراتر از حد گردشدن: {symbol}'
    start_text, end_text = props['Backtesting range'].split('—')
    start = pd.to_datetime(start_text.strip())
    end = pd.to_datetime(end_text.strip())
    file_checks.append({'symbol': symbol, 'closed_trades': len(closed), 'open_trades': reported_open, 'coverage_start': start, 'coverage_end': end, 'first_entry': table['entry_time'].min(), 'reported_pnl_usd': reported_pnl, 'sum_trade_pnl_usd': closed['pnl_usd'].sum(), 'rounding_difference_usd': pnl_difference, 'max_days_between_entries': table['entry_time'].diff().dt.total_seconds().max() / 86400})
    settings.append({'symbol': symbol, **props})
    all_trades.append(table)
trades = pd.concat(all_trades, ignore_index=True)
checks = pd.DataFrame(file_checks)
settings = pd.DataFrame(settings)
expected_currencies = ['AUD', 'CAD', 'CHF', 'EUR', 'GBP', 'NZD', 'USD']
expected_pairs = {frozenset(pair) for pair in itertools.combinations(expected_currencies, 2)}
actual_pairs = {frozenset([s.split(':')[1][:3], s.split(':')[1][3:]]) for s in trades['symbol'].unique()}
assert actual_pairs == expected_pairs
assert checks['symbol'].nunique() == 21
assert not trades['trade_id'].duplicated().any()
for column, expected in {'Timeframe': '1 hour', 'Chart type': 'Candles', 'RSI length (close)': '14', 'Long entry: cross below': '30', 'Short entry: cross above': '70', 'RSI target distance from signal RSI': '17', 'Fixed position quantity (symbol units)': '1000'}.items():
    assert settings[column].astype(str).eq(expected).all(), column
for column in settings.columns:
    if column not in ['symbol', 'Symbol', 'Trading range', 'Backtesting range', 'Currency', 'Tick size']:
        assert settings[column].astype(str).nunique() == 1, f'تنظیم متفاوت: {column}'
trades.to_csv(output_folder / 'merged_trades.csv', index=False, encoding='utf-8-sig')
checks.to_csv(output_folder / 'data_checks.csv', index=False, encoding='utf-8-sig')
settings.to_csv(output_folder / 'source_settings.csv', index=False, encoding='utf-8-sig')

trades = pd.read_csv(output_folder / 'merged_trades.csv', parse_dates=['entry_time', 'exit_time'])
closed = trades[trades['status'] == 'closed'].copy()

def profit_factor(pnl):
    profit = pnl[pnl > 0].sum()
    loss = -pnl[pnl < 0].sum()
    if loss > 0:
        return profit / loss
    return np.inf if profit > 0 else np.nan
pair_rows = []
for symbol, group in closed.groupby('symbol'):
    long = group[group['direction'] == 'long']['pnl_usd']
    short = group[group['direction'] == 'short']['pnl_usd']
    smaller_count = min(len(long), len(short))
    count_ratio = max(len(long), len(short)) / smaller_count if smaller_count else np.inf
    pf_long = profit_factor(long)
    pf_short = profit_factor(short)
    valid_pf = np.isfinite(pf_long) and np.isfinite(pf_short) and (min(pf_long, pf_short) > 0)
    pf_ratio = max(pf_long, pf_short) / min(pf_long, pf_short) if valid_pf else np.nan
    pair_rows.append({'symbol': symbol, 'trades': len(group), 'long_count': len(long), 'short_count': len(short), 'net_pnl_usd': group['pnl_usd'].sum(), 'long_pnl_usd': long.sum(), 'short_pnl_usd': short.sum(), 'pf': profit_factor(group['pnl_usd']), 'pf_long': pf_long, 'pf_short': pf_short, 'count_ratio': count_ratio, 'pf_ratio': pf_ratio, 'at_least_100': len(group) >= 100, 'at_least_800': len(group) >= 800, 'count_balance_ok': count_ratio <= 1.2 + 1e-12, 'pf_balance_ok': bool(valid_pf and pf_ratio <= 1.1 + 1e-12)})
pair_report = pd.DataFrame(pair_rows)
pair_report['rule_1_ok'] = pair_report[['at_least_800', 'pf_balance_ok']].all(axis=1)
pair_report['full_pair_rules_ok'] = pair_report[['rule_1_ok', 'count_balance_ok']].all(axis=1)
pair_report.to_csv(output_folder / 'pair_rules.csv', index=False, encoding='utf-8-sig')
winners = pair_report[pair_report['net_pnl_usd'] > 0]
losers = pair_report[pair_report['net_pnl_usd'] < 0]
flat_count = int((pair_report['net_pnl_usd'] == 0).sum())
mean_profit = winners['net_pnl_usd'].mean()
mean_loss = losers['net_pnl_usd'].abs().mean()
mean_comparison = mean_profit >= mean_loss if len(winners) and len(losers) else None
overall = pd.DataFrame([{'rule': 'حداقل ۱۰۰ معامله برای تک\u200cتک ۲۱ جفت', 'value': int(pair_report['trades'].min()), 'passed': bool(pair_report['at_least_100'].all())}, {'rule': 'حداقل ۱۴ جفت سودده', 'value': len(winners), 'passed': len(winners) >= 14}, {'rule': 'میانگین سود جفت\u200cهای مثبت ≥ میانگین قدرمطلق زیان جفت\u200cهای منفی', 'value': mean_profit / mean_loss if len(losers) and mean_loss else np.nan, 'passed': mean_comparison}])
overall.to_csv(output_folder / 'overall_rules.csv', index=False, encoding='utf-8-sig')
pair_report['long_pct'] = pair_report['long_count'] / pair_report['trades'] * 100
pair_report['short_pct'] = pair_report['short_count'] / pair_report['trades'] * 100
pair_report['rule_1_ok'] = pair_report[['at_least_800', 'count_balance_ok', 'pf_balance_ok']].all(axis=1)
pair_report.to_csv(output_folder / 'pair_rules.csv', index=False, encoding='utf-8-sig')

from IPython.display import display, HTML

def show_table(table):
    view = table.copy()
    view.columns = [str(col).replace('_', ' ').title() for col in view.columns]
    for col in view:
        if view[col].dtype == bool:
            view[col] = view[col].map({True: 'PASS', False: 'FAIL'})
    display(HTML(view.to_html(index=False, border=0, na_rep='—', float_format=lambda value: f'{value:,.2f}')))


print(f"Pairs: {trades['symbol'].nunique()} | Closed trades: {len(closed):,} | Open trades: {(trades['status'] == 'open').sum()}")
print('Validation: unique trade IDs, matched exits, chronological order, trade counts and rounding reconciliation checked.')
print('Independent merged dataset: merged_trades.csv')

Pairs: 21 | Closed trades: 49,520 | Open trades: 4
Validation passed. Independent merged dataset: merged_trades.csv


## 1. Minimum sample size

Requirement: approximately 800 or more closed trades per pair. Counts and directional shares are shown below.

In [2]:
show_table(pair_report[['symbol', 'trades', 'long_count', 'short_count', 'long_pct', 'short_pct', 'at_least_800']])
print(f"Result: {pair_report['at_least_800'].sum()}/21 pairs meet the minimum.")

Symbol,Trades,Long Count,Short Count,Long Pct,Short Pct,At Least 800
FX:AUDCAD,2150,1058,1092,49.21,50.79,PASS
FX:AUDCHF,2329,1210,1119,51.95,48.05,PASS
FX:AUDNZD,2051,1046,1005,51.00,49.00,PASS
FX:AUDUSD,2461,1285,1176,52.21,47.79,PASS
FX:CADCHF,2383,1200,1183,50.36,49.64,PASS
FX:EURAUD,2441,1224,1217,50.14,49.86,PASS
FX:EURCAD,2466,1221,1245,49.51,50.49,PASS
FX:EURCHF,2071,1082,989,52.25,47.75,PASS
FX:EURGBP,2569,1298,1271,50.53,49.47,PASS
FX:EURNZD,2297,1152,1145,50.15,49.85,PASS


Result: 21/21 pairs meet the minimum.


## 2. Long/short count balance

Requirement: the larger directional count must not exceed the smaller by more than 20%. Ratio limit: 1.20.

In [3]:
show_table(pair_report[['symbol', 'long_count', 'short_count', 'count_ratio', 'count_balance_ok']])
print(f"Result: {pair_report['count_balance_ok'].sum()}/21 pairs pass.")

Symbol,Long Count,Short Count,Count Ratio,Count Balance Ok
FX:AUDCAD,1058,1092,1.03,PASS
FX:AUDCHF,1210,1119,1.08,PASS
FX:AUDNZD,1046,1005,1.04,PASS
FX:AUDUSD,1285,1176,1.09,PASS
FX:CADCHF,1200,1183,1.01,PASS
FX:EURAUD,1224,1217,1.01,PASS
FX:EURCAD,1221,1245,1.02,PASS
FX:EURCHF,1082,989,1.09,PASS
FX:EURGBP,1298,1271,1.02,PASS
FX:EURNZD,1152,1145,1.01,PASS


Result: 21/21 pairs pass.


## 3. Long/short profit-factor balance

Requirement: the larger directional PF must not exceed the smaller by more than 10%. Ratio limit: 1.10. Positive net profit in both directions is not imposed as an additional condition.

In [4]:
show_table(pair_report[['symbol', 'pf_long', 'pf_short', 'pf_ratio', 'pf_balance_ok']])
print(f"Result: {pair_report['pf_balance_ok'].sum()}/21 pairs pass.")

Symbol,Pf Long,Pf Short,Pf Ratio,Pf Balance Ok
FX:AUDCAD,1.00,1.32,1.32,FAIL
FX:AUDCHF,0.99,1.05,1.05,PASS
FX:AUDNZD,0.98,1.20,1.23,FAIL
FX:AUDUSD,1.02,1.11,1.09,PASS
FX:CADCHF,0.95,1.06,1.12,FAIL
FX:EURAUD,0.98,1.00,1.02,PASS
FX:EURCAD,1.02,1.08,1.06,PASS
FX:EURCHF,1.14,1.26,1.11,FAIL
FX:EURGBP,1.06,0.91,1.17,FAIL
FX:EURNZD,1.12,0.94,1.19,FAIL


Result: 11/21 pairs pass.


## 4. Preliminary sample requirement

Requirement: at least 100 closed trades on every pair; a total of 2,100 alone is insufficient.

In [5]:
print('Result:', 'PASS' if pair_report['at_least_100'].all() else 'FAIL')
print(f"Minimum per pair: {pair_report['trades'].min():,}. Total closed trades: {len(closed):,}.")

Result: PASS
Minimum per pair: 2,051. Total closed trades: 49,520.


## 5. Number of profitable pairs

Requirement: at least 14 of 21 pairs must have positive net realized profit.

In [6]:
profitability = pair_report[['symbol', 'net_pnl_usd']].copy()
profitability['status'] = np.select([profitability['net_pnl_usd'] > 0, profitability['net_pnl_usd'] < 0], ['Profitable', 'Losing'], default='Breakeven')
show_table(profitability)
print(f"Result: {'PASS' if len(winners) >= 14 else 'FAIL'} — {len(winners)}/21 profitable pairs.")

Symbol,Net Pnl Usd,Status
FX:AUDCAD,386.67,Profitable
FX:AUDCHF,70.45,Profitable
FX:AUDNZD,174.46,Profitable
FX:AUDUSD,242.62,Profitable
FX:CADCHF,3.13,Profitable
FX:EURAUD,-39.66,Losing
FX:EURCAD,230.28,Profitable
FX:EURCHF,517.75,Profitable
FX:EURGBP,-95.99,Losing
FX:EURNZD,129.81,Profitable


Result: PASS — 14/21 profitable pairs.


## 6. Average winning-pair profit versus average losing-pair loss

Requirement: average profit of profitable pairs should be approximately equal to or greater than average absolute loss of losing pairs. No discretionary tolerance is applied.

In [7]:
comparison = pd.DataFrame({'average_winning_pair_profit_usd': [mean_profit], 'average_losing_pair_loss_abs_usd': [mean_loss], 'profit_to_loss_ratio': [mean_profit / mean_loss if mean_loss > 0 else np.nan], 'passed': [mean_comparison]})
show_table(comparison)

Average Winning Pair Profit Usd,Average Losing Pair Loss Abs Usd,Profit To Loss Ratio,Passed
298.79,124.14,2.41,PASS


## 7. Rule assessment conclusion

In [8]:
preliminary_pass = bool(pair_report['at_least_100'].all() and len(winners) >= 14 and mean_comparison)
print('Preliminary 21-pair screen:', 'PASS' if preliminary_pass else 'FAIL')
print(f"Full pair-level conditions (sample size, count balance and PF balance): {pair_report['full_pair_rules_ok'].sum()}/21 pass.")
print('Passing pairs:', ', '.join(pair_report.loc[pair_report['full_pair_rules_ok'], 'symbol'].str.replace('FX:', '', regex=False)))
print('The optimization-exclusion condition is omitted as instructed.')

Preliminary 21-pair screen: PASS
Full pair-level conditions (sample size, count balance and PF balance): 11/21 pass.
Passing pairs: AUDCHF, AUDUSD, EURAUD, EURCAD, GBPAUD, GBPCAD, GBPCHF, GBPNZD, NZDCAD, USDCAD, USDCHF
The optimization-exclusion condition is omitted as instructed.


## 8. Annual realized performance

Profit is assigned to the year of exit. Each pair uses its available history. CADCHF begins on 31 October 2008; the other pairs begin on 15 September 2008. All coverage ends on 20 November 2026. The first and last calendar years are incomplete.

Compact tables show realized performance in USD, first for all pairs combined and then for each pair. A dash means no coverage, not zero profit. Combined totals sum independent backtests; they do not represent a capital-constrained shared account.

In [9]:
checks['coverage_start'] = pd.to_datetime(checks['coverage_start'])
checks['coverage_end'] = pd.to_datetime(checks['coverage_end'])

def performance_by_period(frequency):
    rows = []
    for _, coverage in checks.iterrows():
        symbol = coverage['symbol']
        data = closed[closed['symbol'] == symbol].copy()
        data['period'] = data['exit_time'].dt.to_period(frequency)
        for period in pd.period_range(coverage['coverage_start'], coverage['coverage_end'], freq=frequency):
            pnl = data.loc[data['period'] == period, 'pnl_usd']
            rows.append({
                'symbol': symbol, 'period': str(period),
                'partial_period': bool(coverage['coverage_start'] > period.start_time or coverage['coverage_end'] < period.end_time),
                'closed_trades': len(pnl), 'wins': int((pnl > 0).sum()), 'losses': int((pnl < 0).sum()),
                'breakeven': int((pnl == 0).sum()),
                'gross_profit_usd': pnl[pnl > 0].sum(), 'gross_loss_abs_usd': -pnl[pnl < 0].sum(),
                'net_pnl_usd': pnl.sum(), 'win_rate_pct': (pnl > 0).mean() * 100 if len(pnl) else np.nan,
                'profit_factor': profit_factor(pnl), 'average_trade_usd': pnl.mean() if len(pnl) else np.nan})
    return pd.DataFrame(rows)

yearly = performance_by_period('Y')
monthly = performance_by_period('M')

def combined_performance(report):
    result = report.groupby('period', as_index=False).agg(
        covered_pairs=('symbol', 'nunique'), partial_pairs=('partial_period', 'sum'),
        closed_trades=('closed_trades', 'sum'), wins=('wins', 'sum'), losses=('losses', 'sum'),
        gross_profit_usd=('gross_profit_usd', 'sum'), gross_loss_abs_usd=('gross_loss_abs_usd', 'sum'),
        net_pnl_usd=('net_pnl_usd', 'sum'))
    result['win_rate_pct'] = result['wins'].div(result['closed_trades'].replace(0, np.nan)) * 100
    result['profit_factor'] = result['gross_profit_usd'].div(result['gross_loss_abs_usd'].replace(0, np.nan))
    result.loc[(result['gross_loss_abs_usd'] == 0) & (result['gross_profit_usd'] > 0), 'profit_factor'] = np.inf
    return result

annual_total = combined_performance(yearly)
monthly_total = combined_performance(monthly)
for name, report in [('yearly_per_pair', yearly), ('monthly_per_pair', monthly), ('yearly_all_pairs', annual_total), ('monthly_all_pairs', monthly_total)]:
    report.to_csv(output_folder / (name + '.csv'), index=False, encoding='utf-8-sig')
for report in [yearly, monthly]:
    assert report['closed_trades'].sum() == len(closed)
    actual = report.groupby('symbol')['net_pnl_usd'].sum().sort_index()
    expected = closed.groupby('symbol')['pnl_usd'].sum().sort_index()
    assert np.allclose(actual, expected)

print('Combined annual results (USD):')
show_table(annual_total[['period', 'closed_trades', 'net_pnl_usd', 'win_rate_pct', 'profit_factor', 'covered_pairs', 'partial_pairs']])
for symbol in sorted(yearly['symbol'].unique()):
    print(symbol)
    show_table(yearly.loc[yearly['symbol'] == symbol, ['period', 'closed_trades', 'net_pnl_usd', 'win_rate_pct', 'profit_factor', 'partial_period']])
print('Annual reconciliation: PASS.')

Combined annual results (USD):


Period,Closed Trades,Net Pnl Usd,Win Rate Pct,Profit Factor,Covered Pairs,Partial Pairs
2008,634,123.79,50.00,1.03,21,21
2009,2658,-200.42,49.89,0.98,21,0
2010,2756,475.61,51.49,1.07,21,0
2011,2739,-505.42,49.36,0.94,21,0
2012,2795,-26.40,49.66,0.99,21,0
2013,2875,-464.93,48.17,0.92,21,0
2014,2818,-342.64,48.62,0.93,21,0
2015,2838,506.46,50.92,1.08,21,0
2016,2742,502.13,50.88,1.09,21,0
2017,2783,334.40,52.17,1.08,21,0


FX:AUDCAD


Period,Closed Trades,Net Pnl Usd,Win Rate Pct,Profit Factor,Partial Period
2008,15,20.27,53.33,1.29,PASS
2009,106,37.31,54.72,1.19,FAIL
2010,116,39.15,55.17,1.19,FAIL
2011,116,54.21,56.03,1.25,FAIL
2012,120,5.29,50.83,1.03,FAIL
2013,128,2.55,49.22,1.01,FAIL
2014,124,44.81,57.26,1.30,FAIL
2015,125,26.31,51.20,1.15,FAIL
2016,103,-14.63,45.63,0.90,FAIL
2017,118,30.45,54.24,1.24,FAIL


FX:AUDCHF


Period,Closed Trades,Net Pnl Usd,Win Rate Pct,Profit Factor,Partial Period
2008,34,-42.25,44.12,0.81,PASS
2009,121,-56.38,45.45,0.82,FAIL
2010,134,30.35,51.49,1.10,FAIL
2011,131,-95.43,45.04,0.80,FAIL
2012,158,-49.86,44.30,0.82,FAIL
2013,152,-41.00,46.71,0.88,FAIL
2014,138,14.06,50.72,1.07,FAIL
2015,129,26.36,49.61,1.10,FAIL
2016,123,48.59,53.66,1.25,FAIL
2017,123,-2.35,48.78,0.99,FAIL


... (35 outputs left out)

Period,Closed Trades,Net Pnl Usd,Win Rate Pct,Profit Factor,Partial Period
2008,34,-28.97,44.12,0.83,PASS
2009,132,-56.13,45.45,0.86,FAIL
2010,139,72.93,54.68,1.27,FAIL
2011,120,36.20,52.50,1.16,FAIL
2012,118,32.65,55.08,1.23,FAIL
2013,129,-14.52,49.61,0.91,FAIL
2014,146,-27.11,44.52,0.85,FAIL
2015,147,-1.66,51.02,0.99,FAIL
2016,137,-40.83,45.99,0.85,FAIL
2017,142,-8.86,52.11,0.96,FAIL


FX:USDCHF


Period,Closed Trades,Net Pnl Usd,Win Rate Pct,Profit Factor,Partial Period
2008,34,5.06,47.06,1.04,PASS
2009,124,63.64,53.23,1.22,FAIL
2010,142,30.22,52.11,1.10,FAIL
2011,158,-88.94,47.47,0.80,FAIL
2012,147,11.57,51.02,1.05,FAIL
2013,144,-86.40,45.14,0.70,FAIL
2014,123,-25.52,43.90,0.85,FAIL
2015,139,-26.49,50.36,0.92,FAIL
2016,132,18.32,51.52,1.09,FAIL
2017,128,1.74,49.22,1.01,FAIL


Annual reconciliation: PASS.


## 9. Monthly realized performance

Combined results are grouped by year. Pair-level net P&L is shown in two six-month tables per pair. A dash means no coverage; zero means no realized net profit within the stated coverage. September 2008 and November 2026 are partial months; CADCHF begins in late October 2008.

Results are allocated by exit date. Floating P&L, price gaps and missing candles cannot be reconstructed from trade exports.

In [10]:
print('Combined monthly results (USD):')
for year in sorted(monthly_total['period'].str[:4].unique()):
    print(year)
    show_table(monthly_total.loc[monthly_total['period'].str.startswith(year), ['period', 'closed_trades', 'net_pnl_usd', 'win_rate_pct', 'profit_factor', 'covered_pairs', 'partial_pairs']])
for symbol in sorted(monthly['symbol'].unique()):
    print(symbol + ' — monthly net realized P&L (USD)')
    data = monthly.loc[monthly['symbol'] == symbol].copy()
    data['year'] = data['period'].str[:4]
    data['month'] = data['period'].str[5:7]
    for start, end in [(1, 6), (7, 12)]:
        months = [f'{month:02d}' for month in range(start, end + 1)]
        view = data.pivot(index='year', columns='month', values='net_pnl_usd').reindex(columns=months)
        print('January–June' if start == 1 else 'July–December')
        show_table(view.reset_index())
print('Monthly reconciliation: PASS.')

Combined monthly results (USD):
2008


Period,Closed Trades,Net Pnl Usd,Win Rate Pct,Profit Factor,Covered Pairs,Partial Pairs
2008-09,85,47.17,48.24,1.12,20,20
2008-10,208,-282.39,44.71,0.83,21,1
2008-11,145,357.84,60.69,1.54,21,0
2008-12,196,1.17,48.47,1.00,21,0


2009


Period,Closed Trades,Net Pnl Usd,Win Rate Pct,Profit Factor,Covered Pairs,Partial Pairs
2009-01,211,-656.28,38.39,0.54,21,0
2009-02,197,516.31,64.97,1.92,21,0
2009-03,211,-11.92,50.71,0.98,21,0
2009-04,234,61.08,50.00,1.09,21,0
2009-05,177,-63.97,49.15,0.89,21,0
2009-06,233,70.73,52.79,1.10,21,0
2009-07,236,6.03,50.42,1.01,21,0
2009-08,212,-41.39,49.06,0.93,21,0
2009-09,223,-112.78,45.74,0.82,21,0
2009-10,256,-24.31,49.61,0.97,21,0


2010


Period,Closed Trades,Net Pnl Usd,Win Rate Pct,Profit Factor,Covered Pairs,Partial Pairs
2010-01,181,86.25,56.35,1.21,21,0
2010-02,209,83.83,52.63,1.17,21,0
2010-03,290,49.51,51.38,1.08,21,0
2010-04,252,145.15,54.37,1.30,21,0
2010-05,223,-1.23,48.88,1.00,21,0
2010-06,236,-138.17,44.92,0.82,21,0
2010-07,248,-69.85,48.39,0.91,21,0
2010-08,224,112.82,54.46,1.22,21,0
2010-09,235,118.36,54.47,1.21,21,0
2010-10,181,155.18,57.46,1.38,21,0


... (113 outputs left out)

Year,01,02,03,04,05,06
2008,—,—,—,—,—,—
2009,-13.15,31.52,24.49,-22.84,15.78,14.84
2010,2.31,23.70,5.10,-1.16,-10.79,-5.29
2011,0.72,-11.42,24.39,-11.85,-12.31,11.08
2012,9.90,10.12,-0.09,-0.90,-18.48,-2.64
2013,-17.50,5.64,12.70,-8.13,-21.62,-28.00
2014,-17.28,-10.82,-3.67,-8.92,1.02,-0.97
2015,-17.34,-2.94,-29.79,-12.61,-7.86,19.38
2016,13.86,-9.64,6.21,-4.14,-10.10,-1.58
2017,20.82,8.12,-1.03,6.52,-11.41,8.66


July–December


Year,07,08,09,10,11,12
2008,—,—,-1.09,9.70,21.25,-24.80
2009,6.81,-8.05,-6.16,11.07,2.21,7.12
2010,-3.06,3.15,22.07,6.47,-15.62,3.34
2011,-16.14,-60.29,-24.72,-1.72,1.70,11.62
2012,-4.44,14.61,-3.23,11.55,8.99,-13.82
2013,-10.25,-9.93,-10.66,-1.31,1.71,0.95
2014,-5.03,6.75,0.35,-3.25,18.46,-2.16
2015,15.13,-24.51,9.41,0.66,16.22,7.76
2016,-6.78,-10.87,10.69,3.49,26.80,0.38
2017,6.68,-30.92,-4.83,-5.10,-6.08,10.31


Monthly reconciliation: PASS.
